In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema", "artemzharkov10_silver")

dbutils.widgets.text("gold_catalog", "dbr_dev")
dbutils.widgets.text("gold_schema", "artemzharkov10_gold")


SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

In [0]:
dbutils.widgets.text("TEMP_WATER_FROST", "-1")

dbutils.widgets.text("PRECIP_DRY_MAX", "0")
dbutils.widgets.text("PRECIP_LIGHT_MAX", "2")


TEMP_WATER_FROST = int(dbutils.widgets.get("TEMP_WATER_FROST"))

PRECIP_DRY_MAX = int(dbutils.widgets.get("PRECIP_DRY_MAX"))
PRECIP_LIGHT_MAX = int(dbutils.widgets.get("PRECIP_LIGHT_MAX"))


SILVER_INPUT_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_historical_weather_metrics_for_clustering"
GOLD_OUTPUT_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_weather_clusters"

In [0]:
df_silver = spark.table(SILVER_INPUT_TABLE)
df_gold_weather_claster = (
    df_silver
    .withColumn(
        "temp_claster",
        F.when(F.col("soil_temp") <= TEMP_WATER_FROST, "Frost")
        .when(F.col("soil_temp") > TEMP_WATER_FROST, "Warm")
        .otherwise("normal")
    )
    .withColumn(
        "precipitation_claster",
        F.when(F.col("precipitation") == PRECIP_DRY_MAX, "Dry")
        .when((F.col("precipitation") > PRECIP_DRY_MAX) & (F.col("precipitation") <= PRECIP_LIGHT_MAX), "LightRain")
        .when(F.col("precipitation") > PRECIP_LIGHT_MAX, "HeavyRain")
    )
    .withColumn(
        "weather_claster",
        F.concat_ws("_", F.col("temp_claster"), F.col("precipitation_claster"))
    )
)

(df_gold_weather_claster.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_OUTPUT_TABLE))

In [0]:
# display(df_gold_weather_claster)